# Breast signature analysis -- run + view (inline, parameterised)

Self-contained: load the breast adata, add the cancer gene-set signatures as
spatial-neighborhood factors, then for each target signature **fit + resample + plot** live
in the notebook (no SLURM cache). Change the settings cell to re-run with different
parameters (e.g. `METHOD='ridge', PENALTY=800`). This is the old `signature_analysis`-style
workflow; the cache-backed `signature_analysis_breast.ipynb` viewer is separate.

## Settings

In [ ]:
import sys
from pathlib import Path
import json

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import scanpy as sc
from metab_processing.LinearRegression import signature_helpers as sh
from metab_processing.LinearRegression.signature_configs import CONFIGS
from metab_processing.LinearRegression.build_x import ensure_lognorm_layer

cfg = CONFIGS['breast']

# ---- run parameters (edit + re-run to try different fits) ----
METHOD, PENALTY, STANDARDIZE = 'l1', 0.005, True     # e.g. 'ridge', 800, True
N_SUB = 50
LOWER, UPPER = 15, 85
TOP = 40
PLOTS = ('ratio', 'variance_ratio', 'meansq_variance_ratio')   # which ratio plots to draw

## Load the breast data

In [ ]:
adata = sc.read_h5ad(cfg.adata_path)
ensure_lognorm_layer(adata)
var = set(adata.var_names)
print(adata.n_obs, 'cells |', len(adata.uns.get('x_genes', [])), 'built genes')
print('cell types:', sorted(adata.obs[cfg.cell_type_col].astype(str).unique()))

## Gene sets: targets (predicted) and cancer factors (added as neighborhood predictors)

In [ ]:
library = sh.load_msigdb(cfg.dbver, cfg.categories)

# Target signatures (predicted pseudo-genes): MSigDB targets + manual literal sets.
TARGET_NAMES = list(cfg.target_signatures) + list(cfg.neg_controls)
sets = sh.resolve_sets(TARGET_NAMES, library, var, label='breast targets')
for name, genes in cfg.manual_sets.items():                 # manual (non-MSigDB) sets
    sets[name] = [g for g in genes if g in var]
    print(f'{name}: {len(sets[name])}/{len(genes)} present (manual)')

sh.report_factor_removal(adata, sets)

TARGETS = TARGET_NAMES + list(cfg.manual_sets)

In [ ]:
# Cancer-associated gene sets -> spatial-neighborhood FACTORS (split by cell type) + proximity.
cancer_sets = {n: [g for g in library.get(n, []) if g in var] for n in cfg.cancer_factors}
cancer_sets.update({v: [g for g in gl if g in var] for v, gl in cfg.vessel.items()})
cancer_sets = {k: v for k, v in cancer_sets.items() if v}
sh.add_cancer_factors(adata, cancer_sets, annot_col=cfg.cell_type_col, add_proximity=True)

## Run a signature

`run(sig)` = add the pseudo-gene -> fit (METHOD/PENALTY) on the breast T cells -> resample -> ranked table + the selected ratio plots. `custom='all'` includes the cancer neighborhood factors.

In [ ]:
def run(sig):
    return sh.run_signature(adata, sig, sets, cell_types=list(cfg.t_cell_types),
                            cell_type_col=cfg.cell_type_col, method=METHOD, penalty=PENALTY,
                            standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER,
                            top=TOP, plots=PLOTS)

## T-cell program targets

In [ ]:
run('HALLMARK_IL2_STAT5_SIGNALING')

In [ ]:
run('KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY')

In [ ]:
run('GOBP_T_CELL_MEDIATED_CYTOTOXICITY')

In [ ]:
run('KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY')

In [ ]:
run('HALLMARK_TNFA_SIGNALING_VIA_NFKB')

In [ ]:
run('HALLMARK_INTERFERON_GAMMA_RESPONSE')

In [ ]:
run('HALLMARK_INTERFERON_ALPHA_RESPONSE')

In [ ]:
run('GOBP_CELL_KILLING')

In [ ]:
run('GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY')

In [ ]:
run('GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY')

## Metabolic targets

In [ ]:
run('HALLMARK_HYPOXIA')

In [ ]:
run('HALLMARK_GLYCOLYSIS')

In [ ]:
run('HALLMARK_OXIDATIVE_PHOSPHORYLATION')

In [ ]:
run('HALLMARK_FATTY_ACID_METABOLISM')

In [ ]:
run('HALLMARK_MTORC1_SIGNALING')

## Negative controls

In [ ]:
run('HALLMARK_PANCREAS_BETA_CELLS')

In [ ]:
run('HALLMARK_SPERMATOGENESIS')

In [ ]:
run('MANUAL_ISLET_NEG_CONTROL')